# Notebook de documentacion, tratamiento datos y entrenamiento


## Equipo
- Alumno 1 : Agustín Accurso    
- Alumno 2 : Lautaro Cena

## 1. Introducción y descripción del sistema

Este trabajo práctico implementa un sistema completo de reconocimiento e 
identificación facial, desarrollado en el marco de la materia IA 5.2 Computer Vision.

El sistema es capaz de detectar rostros en imágenes, extraer representaciones 
vectoriales (embeddings), identificar personas registradas y manejar casos de 
desconocidos.

El pipeline sigue el enfoque estándar de sistemas modernos:

    Detección → Alineación → Embeddings → Comparación

La arquitectura fue provista por la cátedra e incluye:
- Backend: FastAPI con procesamiento asincrónico
- Frontend: Gradio
- Base de datos vectorial: PostgreSQL + pgvector
- Modelo: InsightFace (buffalo_l) con ArcFace para extracción de embeddings

Se implementaron las funciones detect_faces, align_face y 
extract_embedding_from_face dentro del servicio de procesamiento facial.

### 1.1 Arquitectura y Pipeline de Procesamiento
A continuación, se detalla el flujo técnico (Pipeline) que sigue el sistema desde la ingesta de la imagen hasta la identificación:

**A. Ingesta y Preprocesamiento**
La imagen se recibe vía el endpoint `/upload` o `/predict`. Se utiliza **OpenCV** (`cv2`) para la carga, operando en **BGR**. Se valida la integridad de los tensores de entrada antes de la detección.

**B. Detección de Rostros**
El método `detect_faces` localiza rostros devolviendo *Bounding Boxes* `[x1, y1, x2, y2]`. 
* **Seguridad:** Se aplica `_clip_xyxy` para asegurar que las coordenadas no excedan los límites de la imagen, evitando errores en los recortes (*slicing*) de NumPy.

**C. Alineación Facial (Geometric Normalization)**
Etapa crítica donde, mediante el método `align_face`, se localizan **Landmarks** y se aplica una **Transformación Afín**. Esto normaliza la rotación y escala, asegurando que el extractor reciba rostros en una orientación constante, minimizando la varianza intra-clase.

**D. Extracción de Embeddings**
El rostro alineado entra a una CNN vía **ONNX Runtime**. 
* **Output:** Un vector (embedding) de **512 dimensiones**. Es una representación en el espacio latente donde la cercanía geométrica implica similitud de identidad.

**E. Comparación (Búsqueda Vectorial)**
Se utiliza la **Similitud Coseno** para comparar el vector de consulta contra el `EmbeddingStore`. Si el score supera el **threshold de 0.55**, se confirma la identidad; de lo contrario, se clasifica como *unknown*.

#### Consideraciones de Diseño
- **Asincronía:** El pipeline es "No bloqueante". Gracias al `TaskManager` y `asyncio`, la API procesa inferencias pesadas sin detener la recepción de nuevas peticiones.
- **Eficiencia:** Se optó por **ONNX Runtime** para la inferencia, logrando una latencia significativamente menor en comparación con modelos estándar de PyTorch.

## 2. Dataset

El dataset fue construido con fotos propias variando las siguientes condiciones:
- Iluminación: natural, artificial, baja luz
- Pose: frente, leve perfil, ángulo desde abajo (selfie)
- Contexto: interior, exterior, espejo
- Vestimenta: distintos looks para evitar que el modelo se apoye en ropa

Las fotos grupales fueron recortadas manualmente para garantizar una sola cara 
por imagen, requisito del endpoint /insert.

No se utilizaron datasets públicos dado que el objetivo del sistema es el 
reconocimiento de personas conocidas con embeddings preentrenados, donde 
no se requiere reentrenamiento del modelo.

| Persona | Cantidad de imágenes | Condiciones | Fuente |
|---------|---------------------|-------------|--------|
| Agus    | 12                  | Selfies, fotos sociales, distintas luces, indoor/outdoor | Fotos personales |
| Silvia    | 13                  | Selfies, fotos sociales, distintas luces, indoor/outdoor | Fotos personales |
| José Luis    | 9                  | Selfies, fotos sociales, distintas luces, indoor/outdoor | Fotos personales |

In [4]:
from pathlib import Path

dataset_path = Path("data")
personas = [p for p in dataset_path.iterdir() if p.is_dir()]

total = 0
for persona in personas:
    imagenes = list(persona.glob("*.jpg")) + list(persona.glob("*.png"))
    print(f"{persona.name}: {len(imagenes)} imágenes")
    total += len(imagenes)

print(f"\nTotal personas: {len(personas)}")
print(f"Total imágenes: {total}")

agus: 12 imágenes
silvia: 13 imágenes

Total personas: 2
Total imágenes: 25


## 3. Preprocesamiento

El preprocesamiento es realizado internamente por InsightFace como parte 
del pipeline de detección y alineación:

- Resize: las caras son normalizadas a 112x112 píxeles (FACE_SIZE=112)
- Normalización: media y desviación estándar aplicadas internamente por ArcFace
- Alineación geométrica: norm_crop transforma la cara usando los 5 keypoints 
  (ojos, nariz, comisuras) para centrar y enderezar el rostro
- No se aplica data augmentation ya que se usan embeddings preentrenados,
  no se entrena el modelo desde cero

Filtrado de calidad: el sistema rechaza imágenes donde no se detecta 
exactamente una cara (register_identity lanza ValueError si hay 0 o más de 1).

## 4. Modelo (Backbone)


### Modelo elegido: InsightFace — buffalo_l (ArcFace + RetinaFace)

Se utilizó el modelo preentrenado buffalo_l de InsightFace, que integra
dos componentes:

- **Detección**: RetinaFace — detecta caras y extrae 5 keypoints faciales
- **Embeddings**: ArcFace (ResNet50) — genera embeddings de 512 dimensiones

### Justificación de la elección

Se optó por un modelo preentrenado sin fine-tuning por las siguientes razones:

1. El objetivo es identificación de personas conocidas, no clasificación 
   de un dataset cerrado. Los embeddings preentrenados generalizan bien 
   a caras no vistas durante el entrenamiento.

2. El hardware disponible es CPU (Ryzen 5 7430U) sin GPU dedicada. 
   Entrenar o hacer fine-tuning desde cero es inviable en este contexto.

3. ArcFace fue entrenado con millones de caras (MS1MV2, ~5.8M imágenes), 
   lo que garantiza representaciones robustas sin necesidad de reentrenamiento.

### Trade-offs

| Criterio             | buffalo_l         | buffalo_sc        |
|----------------------|-------------------|-------------------|
| Precisión            | Alta              | Media             |
| Velocidad en CPU     | Más lenta         | Más rápida        |
| Tamaño del modelo    | ~300MB            | ~30MB             |
| Uso en este TP       | ✓ Elegido         |                   |

Se priorizó precisión sobre velocidad dado que el sistema procesa 
imágenes estáticas, no video en tiempo real.

### Arquitectura interna de ArcFace (ResNet50)

- Input: imagen alineada 112x112 px, 3 canales BGR
- Backbone: ResNet50 con capas convolucionales profundas
- Output: vector de 512 dimensiones (L2-normalizado)
- Loss de entrenamiento: ArcFace Loss (variante de Softmax con margen angular)
  que maximiza la separabilidad entre clases en el espacio esférico

## 5. Embeddings

### Representación vectorial

Cada cara registrada se representa como un vector de 512 dimensiones 
generado por ArcFace. Estos vectores capturan características faciales 
abstractas aprendidas durante el entrenamiento.

### Almacenamiento

Se utiliza PostgreSQL + pgvector (provisto por la cátedra) como base 
vectorial. Cada registro tiene la siguiente estructura:

- id_imagen: identificador único (UUID)
- embedding: vector de 512 floats
- etiqueta: nombre de la persona
- path: ruta de la imagen de referencia guardada
- metadata: información adicional (extensible)

### Búsqueda por similitud

Para identificar una cara, se compara su embedding contra todos los 
registrados usando similitud coseno:

    similitud(a, b) = (a · b) / (||a|| * ||b||)

El resultado es un score entre 0 y 1. Si el mejor score supera el 
threshold (SIMILARITY_THRESHOLD=0.55), se retorna la etiqueta 
correspondiente. Caso contrario se retorna "unknown".

### Visualización de los datos guardados en la BDD

In [8]:
# Con pgvector activo, los embeddings están en PostgreSQL
# Este bloque es solo informativo
print("Base vectorial: PostgreSQL + pgvector")
print("Host: localhost:5432")
print("DB: faces")
print("Los embeddings se almacenan en la tabla de vectores, no en JSON.")

Base vectorial: PostgreSQL + pgvector
Host: localhost:5432
DB: faces
Los embeddings se almacenan en la tabla de vectores, no en JSON.


## 6. Evaluación y métricas

### Para la evaluación del modelo, se implementó un script que inserte todas las imágenes del dataset etiquetadas, de la siguiente manera:


```python
import requests
from pathlib import Path

BASE_URL = "http://localhost:8000"

dataset = {
    "agus": Path("data/agus"),
    "silvia": Path("data/silvia"),
    "jose": Path("data/jose"),
}

for nombre, carpeta in dataset.items():
    fotos = list(carpeta.glob("*.jpg")) + list(carpeta.glob("*.png"))
    for foto in fotos:
        # Paso 1: subir imagen
        with open(foto, "rb") as f:
            upload = requests.post(f"{BASE_URL}/upload", files={"file": f})
        image_path = upload.json()["path"]

        # Paso 2: registrar identidad
        insert = requests.post(f"{BASE_URL}/insert", json={
            "identity": nombre,
            "image_path": image_path,
            "metadata": {}
        })
        print(f"{nombre} | {foto.name} | {insert.status_code}")
```

### Casos individuales

| Imagen | Predicción | Score | Correcto |
|--------|-----------|-------|---------|
| Agus (foto 1) | agus | 0.79 | ✅ |
| Agus (foto 2) | agus | 0.81 | ✅ |
| Silvia (foto 1) | silvia | 0.80 | ✅ |
| Silvia (foto 2) | silvia | 0.78 | ✅ |
| Jose (foto 1) | jose | 0.71 | ✅ |
| Desconocido | unknown | 0.15 | ✅ |

Accuracy casos individuales: 6/6 = 100%

### Casos con múltiples caras

El sistema maneja correctamente imágenes con múltiples personas,
identificando a cada una de forma independiente.

| Imagen | Personas detectadas | Resultado |
|--------|-------------------|---------|
| Agus + 2 desconocidos | agus(0.83), unknown(0.11), unknown(0.10) | ✅ |
| Agus + 2 desconocidos | agus(0.81), unknown(0.06), unknown(0.05) | ✅ |
| Silvia + desconocido | silvia(0.78), unknown(0.13) | ✅ |
| Agus + Jose + Silvia + desconocidos | agus(0.83), jose(0.78), silvia(0.73), unknowns | ✅ |

### Caso límite: variación temporal extrema

Se probó una foto familiar con Agus a los 4 años (actualmente tiene 20).
El sistema reconoció a Jose (0.62) y Silvia (0.62) pero no reconoció 
a Agus, clasificándolo como unknown.

Esto es un comportamiento esperado y correcto: ArcFace genera embeddings 
basados en características faciales adultas. Una diferencia de 16 años 
produce cambios morfológicos que superan la capacidad de generalización 
del modelo, independientemente del threshold configurado.

### Elección del Threshold (0.55)
El umbral de **0.55** fue seleccionado tras realizar pruebas de separabilidad. 
- **Justificación:** Un valor inferior aumentaba la tasa de Falsos Aceptos (personas distintas identificadas como la misma), mientras que un valor superior (ej. 0.70) fallaba ante variaciones naturales de pose o iluminación en las fotos del dataset propio. Este valor garantiza que la distancia en el espacio vectorial sea lo suficientemente discriminativa.

## 7. Conclusiones

### Problemas encontrados y soluciones

* Las coordenadas son correctas en el espacio de la imagen original, el problema es de renderizado en el frontend que no fue modificado.

## 8. Cosas pendientes/ a revisar

* en el item 5, ### Visualización de los datos guardados en la BDD   habria que sacarlo o cambiarlo

* revisar que haya justificaciones para el modelo elegido (analizando costo compt y precision), aclarar si hiciste un *fine-tuning* real o si solo utilizaste un modelo pre-entrenado, ç

* generar un gráfico 2D (usando PCA o t-SNE) para demostrar visualmente que tu sistema logra separar correctamente a cada persona

* **Métricas de evaluación:** Debes armar un set de prueba para incluir una Matriz de Confusión y calcular métricas básicas como *Accuracy* y *Precision*, identificando los falsos positivos y negativos.  (ya está hecho pero falta agregar algunos datos ya armar matriz de confusión)

* ver si aplicar data Augmentation o justificar pq no lo hicimos 

* ampliar un poco la seccion de problemas econtrados. 

* revisar esto q dijeron los profes: El modelo que entrenaron, probablemente pese mas de 100mb, por lo que tendrán que subirlo a algún servicio de almacenamiento ( puede ser Drive de google ) y brindar permisos de solo lectura a cualquier persona con el link. El link deberá quedar documentado en la documentación del Notebook de entrenamiento. -- El modelo deberá subirse a algún servicio de almacenamiento, y brindar un link para su descarga con acceso de solo lectura, el cual debera quedar propiamente documentado en el notebook de entrenamiento.

* dar de acceso de solo lectura a los profes al repositorio para poder acceder al pull request usando nuestros mails. 
